In [2]:
import tensorflow as tf

print("Версия TensorFlow:", tf.__version__)
print("Доступные GPU:", tf.config.list_physical_devices('GPU'))
print("Устройство для вычислений:", tf.test.is_gpu_available())

Версия TensorFlow: 2.14.0
Доступные GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Устройство для вычислений: True


2026-03-09 17:43:34.358499: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-09 17:43:34.359513: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-09 17:43:34.360276: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

In [5]:
import os
import time
import tensorflow_hub as hub
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import kagglehub
from ultralytics import YOLO

DETECTION_THRESHOLD = 0.55

MODEL_URL_rcnn = "https://kaggle.com/models/tensorflow/faster-rcnn-resnet-v1/frameworks/TensorFlow2/variations/faster-rcnn-resnet50-v1-640x640/versions/1"
MODEL_URL_ssd = "https://www.kaggle.com/models/tensorflow/ssd-mobilenet-v2/TensorFlow2/fpnlite-320x320/1"

print("Загрузка моделей...")

start_load_time = time.time()
detector_rcnn = hub.load(MODEL_URL_rcnn).signatures['serving_default']
detector_ssd = hub.load(MODEL_URL_ssd).signatures['serving_default']
detector_yolo = YOLO('yolov8l.pt')

print(f"Модели загружены за {time.time() - start_load_time:.2f} сек")
print("Модели готовы к работе.")

Загрузка моделей...
Модели загружены за 8.16 сек
Модели готовы к работе.


In [6]:
import cv2

VIDEO_DIR = "/home/nikita/images_videos/to_lesson_38/input_videos" 
OUTPUT_DIR = "/home/nikita/images_videos/to_lesson_38/output_videos" 
os.makedirs(OUTPUT_DIR, exist_ok=True)

video_files = [f for f in os.listdir(VIDEO_DIR) if f.endswith(('.mp4', '.avi'))]

In [7]:
COCO_CLASSES = {
    1: 'person', 2: 'bicycle', 3: 'car', 4: 'motorcycle', 5: 'airplane',
    6: 'bus', 7: 'train', 8: 'truck', 9: 'boat', 10: 'traffic light',
    11: 'fire hydrant', 12: 'street sign', 13: 'stop sign', 14: 'parking meter',
    15: 'bench', 16: 'bird', 17: 'cat', 18: 'dog', 19: 'horse', 20: 'sheep',
    21: 'cow', 22: 'elephant', 23: 'bear', 24: 'zebra', 25: 'giraffe',
    26: 'hat', 27: 'backpack', 28: 'umbrella', 29: 'shoe', 30: 'eye glasses',
    31: 'handbag', 32: 'tie', 33: 'suitcase', 34: 'frisbee', 35: 'skis',
    36: 'snowboard', 37: 'sports ball', 38: 'kite', 39: 'baseball bat',
    40: 'baseball glove', 41: 'skateboard', 42: 'surfboard', 43: 'tennis racket',
    44: 'bottle', 45: 'plate', 46: 'wine glass', 47: 'cup', 48: 'fork',
    49: 'knife', 50: 'spoon', 51: 'bowl', 52: 'banana', 53: 'apple',
    54: 'sandwich', 55: 'orange', 56: 'broccoli', 57: 'carrot', 58: 'hot dog',
    59: 'pizza', 60: 'donut', 61: 'cake', 62: 'chair', 63: 'couch',
    64: 'potted plant', 65: 'bed', 66: 'mirror', 67: 'dining table',
    68: 'window', 69: 'desk', 70: 'toilet', 71: 'door', 72: 'tv',
    73: 'laptop', 74: 'mouse', 75: 'remote', 76: 'keyboard', 77: 'cell phone',
    78: 'microwave', 79: 'oven', 80: 'toaster', 81: 'sink', 82: 'refrigerator',
    83: 'blender', 84: 'book', 85: 'clock', 86: 'vase', 87: 'scissors',
    88: 'teddy bear', 89: 'hair drier', 90: 'toothbrush'
}

COCO_CLASSES_yolo = {
    1: 'person', 2: 'bicycle', 3: 'car', 4: 'motorbike', 5: 'aeroplane',
    6: 'bus', 7: 'train', 8: 'truck', 9: 'boat', 10: 'traffic light',
    11: 'fire hydrant', 12: 'stop sign', 13: 'parking meter', 14: 'bench', 15: 'bird',
    16: 'cat', 17: 'dog', 18: 'horse', 19: 'sheep', 20: 'cow',
    21: 'elephant', 22: 'bear', 23: 'zebra', 24: 'giraffe', 25: 'backpack',
    26: 'umbrella', 27: 'handbag', 28: 'tie', 29: 'suitcase', 30: 'frisbee',
    31: 'skis', 32: 'snowboard', 33: 'sports ball', 34: 'kite', 35: 'baseball bat',
    36: 'baseball glove', 37: 'skateboard', 38: 'surfboard', 39: 'tennis racket', 40: 'bottle',
    41: 'wine glass', 42: 'cup', 43: 'fork', 44: 'knife', 45: 'spoon',
    46: 'bowl', 47: 'banana', 48: 'apple', 49: 'sandwich', 50: 'orange',
    51: 'broccoli', 52: 'carrot', 53: 'hot dog', 54: 'pizza', 55: 'donut',
    56: 'cake', 57: 'chair', 58: 'sofa', 59: 'pottedplant', 60: 'bed',
    61: 'diningtable', 62: 'toilet', 63: 'tvmonitor', 64: 'laptop', 65: 'mouse',
    66: 'remote', 67: 'keyboard', 68: 'cell phone', 69: 'microwave', 70: 'oven',
    71: 'toaster', 72: 'sink', 73: 'refrigerator', 74: 'book', 75: 'clock',
    76: 'vase', 77: 'scissors', 78: 'teddy bear', 79: 'hair drier', 80: 'toothbrush'
}


In [8]:
DETECTION_THRESHOLD = 0.55

SKIP_FRAMES = 1  # обрабатывать каждый 2-й кадр
TARGET_SIZE = (640, 640)  
FPS_OUTPUT = 15  # частота кадров сохраняемого видео

In [9]:
def draw_detections(image_bgr, boxes_norm, classes, scores, class_names, threshold, color=(0,255,0)):
    img_copy = image_bgr.copy()
    h, w = img_copy.shape[:2]
    for i in range(len(boxes_norm)):
        if scores[i] >= threshold:
            ymin, xmin, ymax, xmax = boxes_norm[i]
            left = int(xmin * w)
            top = int(ymin * h)
            right = int(xmax * w)
            bottom = int(ymax * h)
            cv2.rectangle(img_copy, (left, top), (right, bottom), color, 2)
            class_id = int(classes[i])
            class_name = class_names.get(class_id, f'unknown_{class_id}')
            label = f"{class_name}: {scores[i]:.2f}"
            cv2.putText(img_copy, label, (left, top-5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    return img_copy

In [10]:
history_rcnn = []
history_ssd = []
history_yolo = []

for video_file in video_files:
    video_path = os.path.join(VIDEO_DIR, video_file)
    cap = cv2.VideoCapture(video_path)
    
    # Информация о видео
    orig_fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"\nОбработка видео: {video_file}")
    print(f"  FPS: {orig_fps}, разрешение: {width}x{height}, всего кадров: {total_frames}")
    
    base_name = os.path.splitext(video_file)[0]
    out_paths = {
        'rcnn': os.path.join(OUTPUT_DIR, f"{base_name}_faster_rcnn.mp4"),
        'ssd': os.path.join(OUTPUT_DIR, f"{base_name}_ssd_mobilenet.mp4"),
        'yolo': os.path.join(OUTPUT_DIR, f"{base_name}_yolov8.mp4")
    }
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_rcnn = cv2.VideoWriter(out_paths['rcnn'], fourcc, FPS_OUTPUT, (width, height))
    out_ssd = cv2.VideoWriter(out_paths['ssd'], fourcc, FPS_OUTPUT, (width, height))
    out_yolo = cv2.VideoWriter(out_paths['yolo'], fourcc, FPS_OUTPUT, (width, height))
    
    frame_count = 0

    temp_rcnn = []
    temp_ssd = []
    temp_yolo = [] 
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_count % (SKIP_FRAMES + 1) != 0:
            frame_count += 1
            continue
        
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        frame_resized = cv2.resize(frame_rgb, TARGET_SIZE)
        
        input_tensor = tf.convert_to_tensor(frame_resized)
        input_tensor = input_tensor[tf.newaxis, ...]
        
        # --- Детекция Faster R-CNN ---
        start_inference_time = time.time()
        result_rcnn = detector_rcnn(input_tensor)
        end_inference_time = time.time()
        inference_time = end_inference_time - start_inference_time
        temp_rcnn.append(inference_time)
        
        result_rcnn = {k: v.numpy() for k, v in result_rcnn.items()}
        boxes_rcnn = result_rcnn['detection_boxes'][0]
        classes_rcnn = result_rcnn['detection_classes'][0].astype(np.int32)
        scores_rcnn = result_rcnn['detection_scores'][0]
        
        # --- Детекция SSD MobileNet ---
        start_inference_time = time.time()
        result_ssd = detector_ssd(input_tensor)
        end_inference_time = time.time()
        inference_time = end_inference_time - start_inference_time
        temp_ssd.append(inference_time)
        
        result_ssd = {k: v.numpy() for k, v in result_ssd.items()}
        boxes_ssd = result_ssd['detection_boxes'][0]
        classes_ssd = result_ssd['detection_classes'][0].astype(np.int32)
        scores_ssd = result_ssd['detection_scores'][0]
        
        # --- Детекция YOLOv8 ---
        start_inference_time = time.time()
        yolo_output = detector_yolo(frame_resized, verbose=False)
        end_inference_time = time.time()
        inference_time = end_inference_time - start_inference_time
        temp_yolo.append(inference_time)
        
        if len(yolo_output[0].boxes) > 0:
            boxes = yolo_output[0].boxes.xyxy.cpu().numpy()
            h_res, w_res, _ = frame_resized.shape
            boxes_norm = boxes / [w_res, h_res, w_res, h_res]
            boxes_yolo = np.stack([boxes_norm[:, 1], boxes_norm[:, 0], boxes_norm[:, 3], boxes_norm[:, 2]], axis=1)
            classes_yolo = yolo_output[0].boxes.cls.cpu().numpy().astype(np.int32) + 1
            scores_yolo = yolo_output[0].boxes.conf.cpu().numpy()
        else:
            boxes_yolo = np.empty((0, 4))
            classes_yolo = np.array([], dtype=np.int32)
            scores_yolo = np.array([])
            
        frame_rcnn = draw_detections(frame, boxes_rcnn, classes_rcnn, scores_rcnn, COCO_CLASSES, 0.6)
        frame_ssd = draw_detections(frame, boxes_ssd, classes_ssd, scores_ssd, COCO_CLASSES, 0.5) 
        frame_yolo = draw_detections(frame, boxes_yolo, classes_yolo, scores_yolo,COCO_CLASSES_yolo, DETECTION_THRESHOLD)          
        
        out_rcnn.write(frame_rcnn)
        out_ssd.write(frame_ssd)
        out_yolo.write(frame_yolo)    

        #просмотр в режиме real-time
        cv2.imshow('YOLO Output', frame_yolo)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        
        frame_count += 1

    history_rcnn.append(temp_rcnn)
    history_ssd.append(temp_ssd)
    history_yolo.append(temp_yolo)
    
    cap.release()
    out_rcnn.release()
    out_ssd.release()
    out_yolo.release()
    cv2.destroyAllWindows()
    print(f"Результаты сохранены:")
    # for model_name, path in out_paths.items():
    #     print(f"  {model_name}: {path}")


Обработка видео: 5064_Chicago_Illinios_1920x1080.mp4
  FPS: 29.97002997002997, разрешение: 1920x1080, всего кадров: 504


2026-03-09 17:44:10.791191: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8700
QFontDatabase: Cannot find font directory /home/nikita/anaconda3/envs/tf-gpu/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nikita/anaconda3/envs/tf-gpu/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nikita/anaconda3/envs/tf-gpu/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nikita/anaconda3/envs/tf-gpu/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships f

Результаты сохранены:

Обработка видео: 6011988_Dog_Animal_1920x1080.mp4
  FPS: 50.0, разрешение: 1920x1080, всего кадров: 733
Результаты сохранены:

Обработка видео: 455390_Athens_Greece_1920x1080.mp4
  FPS: 25.0, разрешение: 1920x1080, всего кадров: 430
Результаты сохранены:


In [11]:
print('время на обработку каждого видео для модели faster r-cnn')
print([round(sum(row),2) for row in history_rcnn], 'секунд')
print('время на обработку каждого видео для модели SSD')
print([round(sum(row),2) for row in history_ssd], 'секунд')
print('время на обработку каждого видео для модели YOLOV8')
print([round(sum(row),2) for row in history_yolo], 'секунд')

время на обработку каждого видео для модели faster r-cnn
[12.19, 16.2, 8.26] секунд
время на обработку каждого видео для модели SSD
[5.44, 7.1, 3.76] секунд
время на обработку каждого видео для модели YOLOV8
[3.34, 4.21, 2.39] секунд


In [13]:
#из hugging_face

from PIL import Image
import torch
from transformers import  AutoImageProcessor, AutoModelForObjectDetection

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используется устройство: {device}")

ckpt = 'yainage90/fashion-object-detection'
image_processor = AutoImageProcessor.from_pretrained(ckpt)
model = AutoModelForObjectDetection.from_pretrained(ckpt).to(device)
model.eval()

# Словарь классов (id -> имя)
id2label = model.config.id2label

Используется устройство: cuda


Loading weights:   0%|          | 0/588 [00:00<?, ?it/s]

In [14]:
def detect_fashion(frame_bgr, threshold=DETECTION_THRESHOLD):

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(frame_rgb)

    with torch.no_grad():
        inputs = image_processor(images=[pil_image], return_tensors="pt").to(device)
        outputs = model(**inputs)
        target_sizes = torch.tensor([[pil_image.size[1], pil_image.size[0]]]).to(device)
        results = image_processor.post_process_object_detection(
            outputs, threshold=threshold, target_sizes=target_sizes
        )[0]

    boxes = results["boxes"].cpu().numpy() 
    classes = results["labels"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    return boxes, classes, scores

In [15]:
for video_file in video_files:
    
    THRESHOLD = 0.5
    
    video_path = os.path.join(VIDEO_DIR, video_file)
    cap = cv2.VideoCapture(video_path)

    orig_fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"\nОбработка видео: {video_file}")
    print(f"  FPS: {orig_fps}, разрешение: {width}x{height}, всего кадров: {total_frames}")

    out_path = os.path.join(OUTPUT_DIR, f"hf_fashion_{video_file}")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, FPS_OUTPUT, (width, height))

    frame_count = 0
    start_inference_time = time.time()
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        boxes, classes, scores = detect_fashion(frame)

        frame_with_boxes = frame.copy()
        for i in range(len(boxes)):
            if scores[i] >= THRESHOLD:
                x1, y1, x2, y2 = boxes[i].astype(int)
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(width, x2)
                y2 = min(height, y2)

                cv2.rectangle(frame_with_boxes, (x1, y1), (x2, y2), (0, 255, 0), 2)

                class_name = id2label.get(int(classes[i]), f"class_{classes[i]}")
                label = f"{class_name}: {scores[i]:.2f}"
                cv2.putText(frame_with_boxes, label, (x1, max(y1-5, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        out.write(frame_with_boxes)

        cv2.imshow('Hugging Face Fashion Detection', frame_with_boxes)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

        frame_count += 1
    
    end_inference_time = time.time()
    inference_time = end_inference_time - start_inference_time
    print('на обработку видео затрачено:', round(inference_time, 2), 'секунд')
    
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"Видео сохранено: {out_path}")

print("\nВсе видео обработаны!")


Обработка видео: 5064_Chicago_Illinios_1920x1080.mp4
  FPS: 29.97002997002997, разрешение: 1920x1080, всего кадров: 504
на обработку видео затрачено: 34.78 секунд
Видео сохранено: /home/nikita/images_videos/to_lesson_38/output_videos/hf_fashion_5064_Chicago_Illinios_1920x1080.mp4

Обработка видео: 6011988_Dog_Animal_1920x1080.mp4
  FPS: 50.0, разрешение: 1920x1080, всего кадров: 733
на обработку видео затрачено: 45.74 секунд
Видео сохранено: /home/nikita/images_videos/to_lesson_38/output_videos/hf_fashion_6011988_Dog_Animal_1920x1080.mp4

Обработка видео: 455390_Athens_Greece_1920x1080.mp4
  FPS: 25.0, разрешение: 1920x1080, всего кадров: 430
на обработку видео затрачено: 25.97 секунд
Видео сохранено: /home/nikita/images_videos/to_lesson_38/output_videos/hf_fashion_455390_Athens_Greece_1920x1080.mp4

Все видео обработаны!
